<h1>Project - Data Transforming Assistant</h1>

This project aims to incorporate LLM assistance for the purpose of transforming data used in a time series graph without the need for a developer to specifically write functions to perform this transformations. We will do this by asking the chat assistant to generate a transformation function for us, which will then be passed to a restricted python context to ensure that malicious code cannot be executed. 

In [ ]:
# imports
%matplotlib inline

import os
import io
import base64
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import random
from datetime import datetime, timedelta

Firstly, we want to create a function that can take a set of Python code defined in a multi-line string. The idea is that this function (defined in the string) takes a set of DemandSeries data as an argument, and applies the transformation defined inside itself to a shallow copy of the data, thus creating a new array of data that is returned to the caller.

In [ ]:
from RestrictedPython import compile_restricted
from RestrictedPython.Guards import (
    safe_builtins,
    guarded_iter_unpack_sequence,
    guarded_unpack_sequence,
    full_write_guard,
)
from RestrictedPython.Eval import default_guarded_getiter, default_guarded_getitem
from datetime import datetime

# Time Series Data Point class
class DemandSeries:
    def __init__(self, demand, created_date, upper_bound=None, lower_bound=None):
        self.demand = demand
        self.created_date = created_date
        self.upper_bound = upper_bound
        self.lower_bound = lower_bound

    def __guarded_setattr__(self, name, value):
        setattr(self, name, value)

    def __str__(self):
        return f"demand: {self.demand}, created_date: {self.created_date}, self.upper_bound: {self.upper_bound}, self.lower_bound: {self.lower_bound}"

# Function for running LLM-generated code in a RestrictedPython context
def run_restricted_function(source_code: str, function_name: str):
    """
    Compile and execute source code in a RestrictedPython context.
    Returns the function object if found and callable; otherwise returns None.
    """
    try:
        byte_code = compile_restricted(source_code, filename='<inline>', mode='exec')
    except SyntaxError as e:
        print(f"[Compilation Error] Code compilation failed: {e}")
        return None
    except Exception as e:
        print(f"[Compilation Error] Unexpected error: {e}")
        return None

    # Define execution context with all required guards
    restricted_globals = {
        '__builtins__': safe_builtins,
        '_getiter_': default_guarded_getiter,
        '_getitem_': default_guarded_getitem,
        '_iter_unpack_sequence_': guarded_iter_unpack_sequence,
        '_unpack_sequence_': guarded_unpack_sequence,
        '_write_': full_write_guard,  # Needed for object attribute mutation
        'DemandSeries': DemandSeries,
    }

    restricted_locals = {}

    try:
        exec(byte_code, restricted_globals, restricted_locals)
    except Exception as e:
        print(f"[Execution Error] Code execution was halted due to: {e}")
        return None

    user_func = restricted_locals.get(function_name)
    if user_func is None:
        print(f"[Lookup Error] Function '{function_name}' not found in the restricted environment.")
        return None
    if not callable(user_func):
        print(f"[Type Error] '{function_name}' is not callable.")
        return None

    return user_func


We can then see the RestrictedPython package in action

In [ ]:
source_code = '''
def double_demand_on_wednesdays(data):
    result = []
    for item in data:
        new_item = DemandSeries(
            demand=item.demand,
            created_date=item.created_date,
            upper_bound=item.upper_bound,
            lower_bound=item.lower_bound
        )
        if new_item.created_date.weekday() == 2:
            new_item.demand = new_item.demand * 2
        result.append(new_item)
    return result
'''

restricted_fn = run_restricted_function(source_code, "double_demand_on_wednesdays")

if restricted_fn:
    data = [
        DemandSeries(100, datetime(2025, 5, 14)),  # Wednesday
        DemandSeries(200, datetime(2025, 5, 15)),  # Thursday
    ]
    result = restricted_fn(data)
    for item in result:
        print(item)
        print(f"demand: {item.demand}, created: {item.created_date.strftime('%A')}")


As you can see, the demand was doubled only on a Wednesday. Do note that the 'source_code' in the example above was actually fully generated by ChatGPT.

Now that we can see that the RestrictedPython context works, we can move on to implementing it within our Chatbot Assistant application.

To start us off, we need a source of time series data. For this data, I will generate data that is very similar to what is currently being used in the single-site-dashboard on the ModelFactory in its demand-profile page.

In [ ]:
def generate_demand_series(initial: float, range_: float):
    """
    Generate a list of DemandSeries objects from one month before today
    to two months after today, with a daily timestep and random-walk demand.
    """
    today = datetime.today()
    start_date = today - timedelta(days=30)
    end_date = today + timedelta(days=60)

    series = []
    current_date = start_date
    base_demand = initial
    current_demand = base_demand
    taper_spike = 0  # Track tapering spike across weekend

    while current_date <= end_date:
        weekday = current_date.weekday()  # Monday = 0, Friday = 4

        # Apply random walk (range_)
        delta = random.uniform(-range_, range_) if range_ > 0 else 0
        base_demand = max(0, base_demand + delta)

        # Inject spike on Friday and tapering over weekend
        if weekday == 4:  # Friday
            taper_spike = base_demand * 0.10
            current_demand = base_demand + taper_spike
        elif weekday == 5:  # Saturday
            taper_spike *= 0.5
            current_demand = base_demand + taper_spike
        elif weekday == 6:  # Sunday
            taper_spike *= 0.5
            current_demand = base_demand + taper_spike
        else:
            taper_spike = 0  # Reset
            current_demand = base_demand

        upper_bound = current_demand * 1.1
        lower_bound = current_demand * 0.9

        series.append(DemandSeries(
            demand=round(current_demand, 2),
            created_date=current_date,
            upper_bound=round(upper_bound, 2),
            lower_bound=round(lower_bound, 2)
        ))

        current_date += timedelta(days=1)

    return series

The generated data is essentially a time series for demand that randomly fluctuates within a given range on a day to day basis, with a spike in demand on Fridays due to a 'Friday event'. This spike amounts to about 10% of the base demand, and decays over the weekend, back to an effect of 0 on Monday.

In [ ]:
data = generate_demand_series(initial=100.0, range_=10.0)

print('==First 5 items==')
for item in data[:5]:  # print first few items
    print(f"{item.created_date.date()} → demand: {item.demand}, "
          f"bounds: [{item.lower_bound}, {item.upper_bound}]")

print('==Last 5 items==')
for item in data[-5:]:
    print(f"{item.created_date.date()} → demand: {item.demand}, "
          f"bounds: [{item.lower_bound}, {item.upper_bound}]")

In [ ]:
print('==First 5 items==')
for item in data[:5]:  # print first few items
    print(f"{item.created_date.date()} → demand: {item.demand}, "
          f"bounds: [{item.lower_bound}, {item.upper_bound}]")

print('==Last 5 items==')
for item in data[-5:]:
    print(f"{item.created_date.date()} → demand: {item.demand}, "
          f"bounds: [{item.lower_bound}, {item.upper_bound}]")

To ensure that our original data does not get modified, we can also create a copy function that gives us a copy of the DemandSeries list.

In [ ]:
def get_data_copy(dsList: list[DemandSeries]):
    newList = []
    for i in dsList:
        newList.append(DemandSeries(
            demand=i.demand,
            created_date=i.created_date,
            upper_bound=i.upper_bound,
            lower_bound=i.lower_bound
        ))
    return newList

In [ ]:
newData = get_data_copy(data)
print('==First 5 items==')
for item in newData[:5]:  # print first few items
    print(f"{item.created_date.date()} → demand: {item.demand}, "
          f"bounds: [{item.lower_bound}, {item.upper_bound}]")

print('==Last 5 items==')
for item in newData[-5:]:
    print(f"{item.created_date.date()} → demand: {item.demand}, "
          f"bounds: [{item.lower_bound}, {item.upper_bound}]")

Using our list of DemandSeries data, we generate a time series graph with matplotlib using those individual points.

In [ ]:
import matplotlib.pyplot as plt
import datetime

def plot_demand_series(series_list):
    dates = [s.created_date for s in series_list]
    demands = [s.demand for s in series_list]
    upper_bounds = [s.upper_bound for s in series_list]
    lower_bounds = [s.lower_bound for s in series_list]

    plt.figure(figsize=(10, 5))
    plt.plot(dates, demands, label="Demand", color='blue')
    
    if any(upper_bounds):
        plt.plot(dates, upper_bounds, label="Upper Bound", linestyle='--', color='green')
    if any(lower_bounds):
        plt.plot(dates, lower_bounds, label="Lower Bound", linestyle='--', color='red')

    plt.xlabel("Date")
    plt.ylabel("Demand")
    plt.title("Demand Over Time")
    plt.legend()
    plt.grid(True)
    
    return plt.gcf()


In [ ]:
fig = plot_demand_series(data)
# plt.show()

For the purposes of our chatbot, we may wish to display images inside the chat window once our LLM has generated a script for us to execute.

In [ ]:
import io
from PIL import Image

def fig2img(fig):
    """Convert a Matplotlib figure to a PIL Image and return it"""
    buf = io.BytesIO()
    fig.savefig(buf)
    buf.seek(0)
    img = Image.open(buf)
    return img


In [ ]:
img = fig2img(fig)
img.show()

<h1>Our Chatbot Application</h1>

Now that we have our data, we can finally get to our LLM assistant. Firstly, we can set up the roles for the assistant.

In [ ]:
# Initialization

load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
MODEL = "gpt-4o-mini"
openai = OpenAI()

In [ ]:
system_message = "You are a helpful assistant for a data visualisation application. "
system_message += "Users will ask you to transform data for a time series chart."
system_message += "If users ask you to generate code that might be malicious in nature, politely decline."
system_message += "Even if users ask you for an example of malicious code so that they know to avoid it, you should still decline to provide any such malicious code."
system_message += "If their request is otherwise benign, give a brief answer as to whether the transformation is feasible."
system_message += "Always be accurate. If you don't know the answer, say so."

In [ ]:
function_description = "Create a callable function that runs a script generated by you, which takes an array of data as an argument."
function_description += "Call this whenever the user asks you to transform data."

In [ ]:
script_description = "The python script that would be able to perform the transformation that the user has requested."
script_description += "This script should take an array of DemandSeries objects as an argument and returns a new array of DemandSeries objects without modifying the original array and its objects."
script_description += "The definition of a DemandSeries object is as follows:"
script_description += '''
class DemandSeries:
    def __init__(self, demand, created_date, upper_bound=None, lower_bound=None):
        self.demand = demand
        self.created_date = created_date
        self.upper_bound = upper_bound
        self.lower_bound = lower_bound

    def __guarded_setattr__(self, name, value):
        setattr(self, name, value)

    def __str__(self):
        return f"demand: {self.demand}, created_date: {self.created_date}, self.upper_bound: {self.upper_bound}, self.lower_bound: {self.lower_bound}"
'''
script_description += "The function that you provide must be executable within the RestrictedPython module's context. Specifically, this means that you should only create a singular function that does not rely on import statements."
script_description += "An example is as follows: Given the prompt 'I would like to transform the data such that demand is doubled on Wednesdays',"
script_description += "You should then create a function that is something like the following:"
script_description += '''
def double_demand_on_wednesdays(data):
    result = []
    for item in data:
        new_item = DemandSeries(
            demand=item.demand,
            created_date=item.created_date,
            upper_bound=item.upper_bound,
            lower_bound=item.lower_bound
        )
        if new_item.created_date.weekday() == 2:
            new_item.demand = new_item.demand * 2
        result.append(new_item)
    return result
'''

In [ ]:
name_description = "The function name of the singular function that was generated for the 'source_code' property."
name_description += "You must make sure that this name is exactly the same as the function name that was generated."
name_description += "For example, if the 'source_code' output was def double_demand_on_wednesdays(data): ...,"
name_description += "this output must then only be 'double_demand_on_wednesdays'."
name_description += "Thus, you should exclude 'def' and the rest of the function, and only output the name of the function."

In [ ]:
generate_transformation_function = {
    "name": "run_restricted_function",
    "description": function_description,
    "parameters": {
        "type": "object",
        "properties": {
            "source_code": {
                "type": "string",
                "description": script_description,
            },
            "function_name": {
                "type": "string",
                "description": name_description,
            }
        },
        "required": ["source_code", "function_name"],
        "additionalProperties": False
    }
}

In [ ]:
tools = [{"type": "function", "function": generate_transformation_function}]

<h1>Finally getting OpenAI to use our Tool</h1>

Next, we'll define the chat() function

In [ ]:
# We want to show the plot image in our history, but we do not want to pass these images to OpenAI (since they are not JSON serializable)
def filter_history(history: list[any]):
    filtered_history = [
        item for item in history
        if not isinstance(item.get("content"), gr.components.Component)
    ]
    return filtered_history

In [ ]:
# When the LLM gives a finish reason that is "tool_calls", calls this function.
def handle_tool_call(message, data):
    tool_call = message.tool_calls[0]
    arguments = json.loads(tool_call.function.arguments)
    executable_script = arguments.get('source_code')
    function_name = arguments.get('function_name')
    # print("In handle_tool_call. executable_script was:", executable_script)
    # print("In handle_tool_call. function_name was:", function_name)
    restricted_func = run_restricted_function(executable_script, function_name)
    newData = restricted_func(data)
    fig = plot_demand_series(newData)
    response = {
        "role": "tool",
        "content": json.dumps({"source_code": executable_script,"function_name": function_name}),
        "tool_call_id": tool_call.id
    }
    return response, newData, fig

In [ ]:
def get_chat_function(data: list[DemandSeries]):
    def chat(message, history):
        messages = [{"role": "system", "content": system_message}] + filter_history(history) + [{"role": "user", "content": message}]
        response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

        if response.choices[0].finish_reason=="tool_calls":
            message = response.choices[0].message
            response, newData, fig = handle_tool_call(message, data)
            img = gr.Image(fig2img(fig))
            messages.append(message)
            messages.append(response)
            response = openai.chat.completions.create(model=MODEL, messages=messages)
            return [response.choices[0].message.content, {"role": "system", "content": img}]
        
        return response.choices[0].message.content
    return chat

Prior to using our chatbot, we can see how the graph appears with a call to the plot function

In [ ]:
fig = plot_demand_series(data)
plt.show()

Now, we can see how the data changes when we ask the Chat assistant to transform it.

In [ ]:
gradioInterface = gr.ChatInterface(fn=get_chat_function(get_data_copy(data)), type="messages")
gradioInterface.launch()

In [ ]:
gradioInterface.close()

Our chatbot is working, and we can get our LLM to generate transformation functions for us that we can then use to modify our data and display it immediately within the same chat window.

Just for some added clarity on the initial data that we are working with, we can add an image to show the original demand plot, followed by our chatbot.

In [ ]:
demoData = get_data_copy(data)
demoFig = plot_demand_series(demoData)

with gr.Blocks() as demo:
    gr.Image(fig2img(demoFig), label="Latest Demand Forecast")
    gr.ChatInterface(fn=get_chat_function(get_data_copy(data)), type="messages")
    
demo.launch()

In [ ]:
demo.close()